In [39]:
import requests
import os
from dotenv import load_dotenv

import pandas as pd
import json
import matplotlib.pyplot as plt
import plotly.express as px
import nbformat
from datetime import datetime

url_base="https://api.stlouisfed.org"
load_dotenv()
api_key = os.getenv("API_KEY")
custom_header = {
    "Authorization": f"Bearer {api_key}"
}

DESIRED_START_DATE = "2000-01-01"

In [2]:
response = requests.get(url_base)

In [3]:
#IDs for Leading Economic Indicators
series_dict = {
    "AvgWeeklyHoursManufacturing": "AWHAEMAN",
    "AvgWeeklyClaimsUI": "ICSA",
    "ManufacturersNewOrdersConsumerGoodsAndMat": "ACOGNO",
    "ManufacturersNewOrdersNondefenseCapital": "NEWORDER",
    "ISMNewOrdersIndex": "UMTMNO",
    "BuildingPermitsNewPrivateHousing": "PERMIT",
    "DowJonesIndustrialAverage": "DJIA",
    "LeadingCreditIndex": "TOTALSL",
    "InterestRateSpread": "T10YFF",
    "AverageConsumerExpectations": "CSCICP03USM665S"
}

#Formal, longer names... MUST have same keys as above dictionary!
series_names = {
    "AvgWeeklyHoursManufacturing": "Average Weekly Hours in Manufacturing",
    "AvgWeeklyClaimsUI": "Average Weekly Claims for Unemployment Insurance",
    "ManufacturersNewOrdersConsumerGoodsAndMat": "Manufacturers\' New Orders, Consumer Goods and Materials",
    "ManufacturersNewOrdersNondefenseCapital": "Manufacturers\' New Orders, Nondefense Capital Goods",
    "ISMNewOrdersIndex": "ISM New Orders Index",
    "BuildingPermitsNewPrivateHousing": "Building Permits for New Private Housing",
    "DowJonesIndustrialAverage": "Dow Jones Industrial Average (DJIA)",
    "LeadingCreditIndex": "Total Consumer Credit Owned and Securitized",
    "InterestRateSpread": "Interest Rate Spread, 10-Year Treasury Bond Yield Minus Federal Funds Rate",
    "AverageConsumerExpectations": "Average Consumer Expectations - Composite Leading Indicators: Composite Consumer Confidence Amplitude Adjusted for United States"
}

In [4]:
#Series needs to be the series code, comes from series_dict
#Returns the requested time series, or empty array if failed the pull request
def get_data_series(series):
    #Make request
    req_url=f"https://api.stlouisfed.org/fred/series/observations?series_id={series}&api_key={api_key}&file_type=json"
    response = requests.get(req_url, headers=custom_header)
    if response.status_code == 200:
        # 3. Parse JSON into a Python dictionary
        data = response.json()
        series = []
        for observation in data['observations']:
            series.append({'date': observation['date'],
                           'value': observation['value']})
        return series
    else:
        print(f"Error: {response.status_code}")
        return []

In [6]:
def get_recession_area():
    #Make request
    req_url=f"https://api.stlouisfed.org/fred/series/observations?series_id=USREC&api_key={api_key}&file_type=json"
    response = requests.get(req_url, headers=custom_header)
    if response.status_code == 200:
        # 3. Parse JSON into a Python dictionary
        data = response.json()
        series = []
        for observation in data['observations']:
            series.append({'date': observation['date'],
                           'value': observation['value']})
        return series
    else:
        print(f"Error: {response.status_code}")
        return []

In [9]:
t_df = get_recession_area()
# Convert to a pandas DataFrame
rec_df = pd.DataFrame(t_df)
rec_df['date'] = pd.to_datetime(rec_df['date'], errors="coerce")
rec_df['value'] = pd.to_numeric(rec_df['value'], errors="coerce")
rec_df.head()

,date,value
0,1854-12-01,1
1,1855-01-01,0
2,1855-02-01,0
3,1855-03-01,0
4,1855-04-01,0


In [32]:
#Print interactive graphs all at once!

def get_value(row):
    # If date is in rec_df
    try:
        return rec_df[rec_df['date']==row['date']].iloc[0]['value']
    except:
        return 0

In [ ]:
for series in series_dict.keys():
    try:
        t_df = get_data_series(series_dict[series])
        # Convert to a pandas DataFrame
        df = pd.DataFrame(t_df)
        df['date'] = pd.to_datetime(df['date'], errors="coerce")
        df['value'] = pd.to_numeric(df['value'], errors="coerce")
        df['recession'] = df.apply(get_value, axis=1)
        
        df.head()
        #df.plot(x='date',y='value', title='Average Weekly Claims for Unemployment Insurance')

        fig = px.line(df, x='date', y='value', title=f"{series_names[series]} - FRED Series {series_dict[series]}")

        #Shade recession areas
        for i in range(len(df) - 1):
            current_category = df['recession'].iloc[i]
            next_category = df['recession'].iloc[i+1]
            recession_start = df['date'].iloc[0]
            if current_category != next_category:
                if(current_category==1):
                    fig.add_vrect(
                        #Want start to be start of recessionary period
                        x0=df['date'].iloc[i],
                        x1=df['date'].iloc[i+1],
                        fillcolor='lightgreen',
                        opacity=0.3,
                        line_width=0,
                    )
                elif(current_category==0):
                    recession_start=df['date'].iloc[i+1]
        #fig.update_xaxes(
        #    range=[DESIRED_START_DATE, datetime.today().strftime('%Y-%m-%d')] # Set the start and end dates as strings or datetime objects
        #)
        fig.show()
    except Exception as e:
        print(e)

Error: 500
'date'
